# EWS Fraud Detection - Tahap 3B: Perhitungan M-Score & Trend Features (Tanpa DEPI)

Notebook ini menghitung indeks Beneish M-Score (tanpa kontribusi DEPI karena nilai penyusutan yang terlalu rendah/tidak signifikan pada model awal), menentukan tingkat risiko (fraud flag), dan membuat fitur tren finansial yang telah dibersihkan menggunakan teknik **Winsorization**.

### Variabel Tambahan yang Dihitung:
1. **DEPI** (Penyusutan - dihitung tapi koefisien diset ke 0 pada formula akhir)
2. **SGAI** (Beban Umum & Administrasi)
3. **TATA** (Akrual Total terhadap Total Aset)
4. **M-Score** (Tanpa komponen DEPI)
5. **Trend Features** (`revenue_growth`, `asset_growth`, `net_income_growth_assets`)
6. **CFO Quality flag**

In [1]:
import os
import pandas as pd
import numpy as np

in_path = "Financial_Metrics_With_Components.xlsx"
out_path = "Fraud_Dataset_Final.xlsx"

def safe_divide(num, denom):
    with np.errstate(divide='ignore', invalid='ignore'):
        res = num / denom
        res = np.where(np.isfinite(res), res, np.nan)
    return res

## 1. Memuat Data & Membuat Fitur Lag untuk Penyusutan & Beban SG&A

In [2]:
df = pd.read_excel(in_path)

# Membuat lag
df['depreciation_lag'] = df.groupby('kode')['depreciation'].shift(1)
df['selling_expense_lag'] = df.groupby('kode')['selling_expense'].shift(1)
df['ga_expense_lag'] = df.groupby('kode')['ga_expense'].shift(1)
df['net_income_lag'] = df.groupby('kode')['net_income'].shift(1)

print(f"Data sukses dimuat. Shape: {df.shape}")

Data sukses dimuat. Shape: (2466, 40)


## 2. Menghitung DEPI, SGAI, dan TATA

In [3]:
# 1. DEPI (Depreciation Index)
dep_t = np.abs(df['depreciation'].fillna(0))
dep_lag = np.abs(df['depreciation_lag'].fillna(0))
dep_rate_t = safe_divide(dep_t, np.abs(df['ppe'].fillna(0)) + dep_t)
dep_rate_lag = safe_divide(dep_lag, np.abs(df['ppe_lag'].fillna(0)) + dep_lag)
df['depi_raw'] = safe_divide(dep_rate_lag, dep_rate_t)
df['depi'] = df['depi_raw'].fillna(1.0).clip(lower=0.1, upper=10.0)

# 2. SGAI (SG&A Index)
sga_t = np.abs(df['selling_expense'].fillna(0)) + np.abs(df['ga_expense'].fillna(0))
sga_lag = np.abs(df['selling_expense_lag'].fillna(0)) + np.abs(df['ga_expense_lag'].fillna(0))
ratio_sga_t = safe_divide(sga_t, df['revenue'])
ratio_sga_lag = safe_divide(sga_lag, df['revenue_lag'])
df['sgai_raw'] = safe_divide(ratio_sga_t, ratio_sga_lag)
df['sgai'] = df['sgai_raw'].fillna(1.0).clip(lower=0.1, upper=10.0)

# 3. TATA (Total Accruals to Total Assets)
df['tata_raw'] = safe_divide(df['net_income'] - df['cfo'].fillna(0), df['total_assets'])
df['tata'] = df['tata_raw'].clip(lower=-1.0, upper=1.0)

print("Perhitungan DEPI, SGAI, dan TATA selesai.")

Perhitungan DEPI, SGAI, dan TATA selesai.


## 3. Menghitung Beneish M-Score (Tanpa DEPI) & Fraud Flags

In [4]:
# Beneish M-Score Formula (DEPI coefficient = 0)
df['m_score'] = (
    -4.84 
    + 0.920 * df['dsri'].fillna(1.0) 
    + 0.528 * df['gmi'].fillna(1.0) 
    + 0.404 * df['aqi'].fillna(1.0) 
    + 0.892 * df['sgi'].fillna(1.0) 
    - 0.172 * df['sgai'].fillna(1.0) 
    + 4.679 * df['tata'].fillna(0.0) 
    - 0.327 * df['lvgi'].fillna(1.0)
)

# Set M-Score ke NaN jika data mentah kunci tidak tersedia
critical_missing = df['dsri'].isnull() | df['gmi'].isnull() | df['aqi'].isnull() | df['sgi'].isnull()
df.loc[critical_missing, 'm_score'] = np.nan

# Klasifikasi Risiko Fraud (-1.78 sebagai threshold)
df['fraud_flag'] = np.where(df['m_score'] > -1.78, 'High Risk', 'Low Risk')
df.loc[df['m_score'].isnull(), 'fraud_flag'] = np.nan

print(f"Rasio High Risk: {(df['fraud_flag'] == 'High Risk').sum() / df['m_score'].notnull().sum() * 100:.2f}%")

Rasio High Risk: 20.39%


## 4. Rekayasa Fitur Tren & CFO Quality (Winsorization 1% & 99%)
Menerapkan Winsorization untuk membersihkan outliers ekstrem.

In [5]:
# Trend Features
df['revenue_growth_raw'] = safe_divide(df['revenue'] - df['revenue_lag'], df['revenue_lag'])
df['asset_growth_raw'] = safe_divide(df['total_assets'] - df['total_assets_lag'], df['total_assets_lag'])
df['net_income_growth_assets_raw'] = safe_divide(df['net_income'] - df['net_income_lag'], df['total_assets_lag'])

for col in ['revenue_growth', 'asset_growth', 'net_income_growth_assets']:
    lower = df[f'{col}_raw'].quantile(0.01)
    upper = df[f'{col}_raw'].quantile(0.99)
    df[col] = df[f'{col}_raw'].clip(lower=lower, upper=upper)

# CFO Quality
df['cfo_to_net_income_raw'] = safe_divide(df['cfo'], df['net_income'])
lower_cfo = df['cfo_to_net_income_raw'].quantile(0.01)
upper_cfo = df['cfo_to_net_income_raw'].quantile(0.99)
df['cfo_to_net_income'] = df['cfo_to_net_income_raw'].clip(lower=lower_cfo, upper=upper_cfo)

df['cfo_quality_flag'] = np.where((df['net_income'] > 0) & (df['cfo'] <= 0), 'Low Quality', 'Normal')
df.loc[df['net_income'].isnull() | df['cfo'].isnull(), 'cfo_quality_flag'] = np.nan

# Ekspor
df.to_excel(out_path, index=False)
print(f"Dataset sukses diekspor ke '{out_path}'. Shape: {df.shape}")

Dataset sukses diekspor ke 'Fraud_Dataset_Final.xlsx'. Shape: (2466, 57)
